In [ ]:
try:
  from sklearnex import patch_sklearn # speed up sklearn if cpu is intel
  patch_sklearn()
except ImportError as sklearnex_not_installed:
  print("sklearnex not installed, use default sklearn instead")
  print("if you want to use sklearn, please refer to https://pypi.org/project/scikit-learn-intelex/")
  

from TrainingData import load_KTH
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn import svm, preprocessing
from sklearn.pipeline import make_pipeline
import pandas as pd

In [ ]:
jogging: str = "walking" # "" (empty string), "walking" or "running"
training_data = load_KTH()
if jogging:
  training_data.label = training_data.label.str.replace("jogging", jogging)
  target_filename = f"jogging={jogging}"
else:
  training_data = training_data.apply(lambda row: row[training_data['label'].isin(['walking','running'])])
  target_filename = "no_jogging"
print(f"{target_filename=}")
training_data.groupby('label').count()

<img src="images/Pose Landmark Model.png"><img>

from https://github.com/google/mediapipe/blob/master/docs/solutions/pose.md

In [ ]:
unneeded_data = ["filename", "label",'NOSE_x', 'NOSE_y', 'LEFT_EYE_INNER_x', 'LEFT_EYE_INNER_y',
       'LEFT_EYE_x', 'LEFT_EYE_y', 'LEFT_EYE_OUTER_x', 'LEFT_EYE_OUTER_y',
       'RIGHT_EYE_INNER_x', 'RIGHT_EYE_INNER_y', 'RIGHT_EYE_x', 'RIGHT_EYE_y',
       'RIGHT_EYE_OUTER_x', 'RIGHT_EYE_OUTER_y', 'LEFT_EAR_x', 'LEFT_EAR_y',
       'RIGHT_EAR_x', 'RIGHT_EAR_y', 'MOUTH_LEFT_x', 'MOUTH_LEFT_y',
       'MOUTH_RIGHT_x', 'MOUTH_RIGHT_y','LEFT_PINKY_x', 'LEFT_PINKY_y',
       'RIGHT_PINKY_x', 'RIGHT_PINKY_y', 'LEFT_INDEX_x', 'LEFT_INDEX_y',
       'RIGHT_INDEX_x', 'RIGHT_INDEX_y', 'LEFT_THUMB_x', 'LEFT_THUMB_y',
       'RIGHT_THUMB_x', 'RIGHT_THUMB_y','LEFT_HEEL_x', 'LEFT_HEEL_y',
       'RIGHT_HEEL_x', 'RIGHT_HEEL_y', 'LEFT_FOOT_INDEX_x',
       'LEFT_FOOT_INDEX_y', 'RIGHT_FOOT_INDEX_x', 'RIGHT_FOOT_INDEX_y',]

In [ ]:
le = preprocessing.LabelEncoder()
training_data['Encoded_label'] = le.fit_transform(training_data.label)
training_data = training_data.drop(unneeded_data, axis=1)
training_data.head()

In [ ]:
le.classes_, le.transform(le.classes_)

In [ ]:
X, y = training_data.iloc[:, :-1],  training_data.Encoded_label
X_train, X_test, y_train, y_test = train_test_split(X, y)

In [ ]:
print(X_train.shape)
X_train.head(1)

In [ ]:
clf = svm.SVC()

In [ ]:
clf.fit(X_train.values, y_train.values)

In [ ]:
clf.score(X_test.values, y_test.values)

In [ ]:
# code from https://www.jcchouinard.com/confusion-matrix-in-scikit-learn/
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import confusion_matrix

y_pred = clf.predict(X_test.values)
cm = confusion_matrix(le.inverse_transform(y_test), le.inverse_transform(y_pred), labels=le.inverse_transform(clf.classes_))
color = 'white'
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.inverse_transform(clf.classes_))
disp.plot()
plt.show()

# Saving the trained model

In [ ]:
import joblib
from pathlib import Path
joblib.dump(clf, str(Path('models')/'action'/f'pose_action_classifier_{target_filename}.pkl'))
joblib.dump(le, str(Path('models')/'label'/'label_encoder.pkl'))

# loading the saved model
```python
# Load the model from the file
pose_action_classifier = joblib.load('filename.pkl')
label_encoder = joblib.load('filename2.pkl')

# Use the loaded model to make predictions
y_pred = pose_action_classifier.predict(X_test)
action_pred = label_encoder.inverse_transform(y_pred)

```